# Import and Manipulate the Truss Dataset

This notebook is a compact tutorial for loading a selected part of the JSON dataset, reusing the processed PyG cache, filtering graphs with `pre_filter`, and scaling results with either `pre_transform` or runtime `transform`.

In [1]:
from pathlib import Path
import math
import sys

import pandas as pd
import torch
from torch_geometric.data import Data

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Run this notebook from trussgraph_merged or trussgraph_merged/notebooks.")

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from dataset import _TrussJsonDataset

DATA_ROOT = PROJECT_ROOT / "data"
print("Project root:", PROJECT_ROOT)

Project root: root


## The provided base class `_TrussJsonDataset` 

`_TrussJsonDataset` is the base importer. It already knows how to:

- find JSON files under `data/raw/<Typology>/p_<panels>_seed_<seed>/`
- filter by typology, panels, seeds, and sample limits
- download missing selected typology folders from Zenodo
- parse each JSON into common tensors: coordinates, loads, displacements, member connectivity, lengths, and axial forces
- cache the processed PyG dataset under `data/processed`

To create a dataset for a new use case, inherit from `_TrussJsonDataset` and implement only `build_data(common, typology_id, panels, seed, group_id)`. The `common` dictionary contains the parsed raw quantities. Your subclass decides what becomes `x`, `edge_attr`, `y`, and which extra fields should be kept.

Bellow is the base importer structure and the object's `common` keys that are the fields available for use inside `build_data(...)` when you create a subclass.

In [2]:
import inspect
import json

from dataset import _parse_common_graph, _select_json_paths

print("_TrussJsonDataset constructor")
print(inspect.signature(_TrussJsonDataset.__init__))

print("""
    Available keys in common:
    coordinates: tensor shape=(15, 2), dtype=torch.float32
    displacement: tensor shape=(15, 2), dtype=torch.float32
    applied_load: tensor shape=(15, 2), dtype=torch.float32
    free_mask: tensor shape=(15,), dtype=torch.bool
    valency: tensor shape=(15,), dtype=torch.float32
    member_index: tensor shape=(2, 27), dtype=torch.int64
    member_length: tensor shape=(27,), dtype=torch.float32
    axial_force: tensor shape=(27,), dtype=torch.float32
    graph_id: int = 1439
    num_members: int = 27
""")

_TrussJsonDataset constructor
(self, root: 'str | Path' = 'data', *, typologies: 'Optional[Sequence[str]]' = None, panels: 'Optional[Sequence[int]]' = None, seeds: 'Optional[Sequence[int]]' = None, max_seeds_per_panel: 'Optional[int]' = None, max_designs_per_seed: 'Optional[int]' = None, sampling_seed: 'int' = 42, plane: 'str' = 'xz', cache_name: 'str', download_if_missing: 'bool' = True, transform=None, pre_transform=None, pre_filter=None, force_reload: 'bool' = False) -> 'None'

    Available keys in common:
    coordinates: tensor shape=(15, 2), dtype=torch.float32
    displacement: tensor shape=(15, 2), dtype=torch.float32
    applied_load: tensor shape=(15, 2), dtype=torch.float32
    free_mask: tensor shape=(15,), dtype=torch.bool
    valency: tensor shape=(15,), dtype=torch.float32
    member_index: tensor shape=(2, 27), dtype=torch.int64
    member_length: tensor shape=(27,), dtype=torch.float32
    axial_force: tensor shape=(27,), dtype=torch.float32
    graph_id: int = 1439
 

## How `_TrussJsonDataset` Is Meant to Be Used

The small `TrussBaseDataset` below keeps the raw physical quantities with very little processing, making it useful for tutorials and data inspection.

In [3]:
class TrussBaseDataset(_TrussJsonDataset):
    node_feature_names = (
        "valency", "is_free", "x", "z", "force_x", "force_z", "ux", "uz"
    )
    edge_feature_names = ("length", "axial_force")
    member_feature_names = ("length", "axial_force")
    target_names = ()
    process_version = 1

    def __init__(self, *args, cache_name="tutorial_base", **kwargs):
        super().__init__(*args, cache_name=cache_name, **kwargs)

    def build_data(self, common, *, typology_id, panels, seed, group_id):
        member_index = common["member_index"]
        member_attr = torch.stack(
            [common["member_length"], common["axial_force"]], dim=-1
        )
        x = torch.cat(
            [
                common["valency"].unsqueeze(-1),
                common["free_mask"].float().unsqueeze(-1),
                common["coordinates"],
                common["applied_load"],
                common["displacement"],
            ],
            dim=-1,
        )

        return Data(
            x=x,
            edge_index=torch.cat([member_index, member_index.flip(0)], dim=1),
            edge_attr=torch.cat([member_attr, member_attr], dim=0),
            member_index=member_index,
            member_attr=member_attr,
            typology_id=torch.tensor([typology_id]),
            panels=torch.tensor([panels]),
            seed=torch.tensor([seed]),
            group_id=torch.tensor([group_id]),
            graph_id=torch.tensor([common["graph_id"]]),
            num_members=torch.tensor([common["num_members"]]),
        )

## Select a Small Subset

The first construction downloads the selected typology if it is missing, processes JSON files, and writes a cache. Keep the limits small while experimenting.

In [4]:
TYPOLOGIES = ["Fink"]
PANELS = [7]
SEEDS = [7]
MAX_DESIGNS_PER_SEED = 25
PLANE = "xz"
CACHE_NAME = "tutorial_base_fink_p7_seed7"

In [5]:
dataset = TrussBaseDataset(
    root=DATA_ROOT,
    typologies=TYPOLOGIES,
    panels=PANELS,
    seeds=SEEDS,
    max_designs_per_seed=MAX_DESIGNS_PER_SEED,
    plane=PLANE,
    cache_name=CACHE_NAME,
    force_reload=False,
)

print(f"Graphs: {len(dataset):,}")
print("Processed cache:", dataset.processed_paths[0])
print("Features:", dataset.metadata["node_feature_names"], dataset.metadata["member_feature_names"])

Graphs: 25
Processed cache: root\data\processed\tutorial_base_fink_p7_seed7_c713da5e52da.pt
Features: ('valency', 'is_free', 'x', 'z', 'force_x', 'force_z', 'ux', 'uz') ('length', 'axial_force')


## Reuse the Processed Cache

Run this cell after the previous one. Because the constructor arguments and `cache_name` are identical, PyG loads the processed file instead of parsing JSON again.

In [6]:
cached_dataset = TrussBaseDataset(
    root=DATA_ROOT,
    typologies=TYPOLOGIES,
    panels=PANELS,
    seeds=SEEDS,
    max_designs_per_seed=MAX_DESIGNS_PER_SEED,
    plane=PLANE,
    cache_name=CACHE_NAME,
    force_reload=False,
)

print(f"Graphs loaded: {len(cached_dataset):,}")
print("Cache reused:", cached_dataset.processed_paths[0])

Graphs loaded: 25
Cache reused: root\data\processed\tutorial_base_fink_p7_seed7_c713da5e52da.pt


## Inspect the Imported Graphs

For this base representation:

- `x[:, 0]` is valency
- `x[:, 1]` is `is_free`
- `x[:, 2:4]` are coordinates
- `x[:, 4:6]` are applied loads
- `x[:, 6:8]` are displacements
- `member_attr[:, 0]` is member length
- `member_attr[:, 1]` is axial force

In [7]:
graph = cached_dataset[0]
print(graph)

rows = []
for item in cached_dataset:
    rows.append({
        "graph_id": int(item.graph_id),
        "panels": int(item.panels),
        "seed": int(item.seed),
        "nodes": item.num_nodes,
        "members": int(item.num_members),
        "min_length": float(item.member_attr[:, 0].min()),
    })

display(pd.DataFrame(rows).head())

Data(x=[15, 8], edge_index=[2, 54], edge_attr=[54, 2], member_index=[2, 27], member_attr=[27, 2], typology_id=[1], panels=[1], seed=[1], group_id=[1], graph_id=[1], num_members=[1])


,graph_id,panels,seed,nodes,members,min_length
0,240,7,7,15,27,1.103244
1,379,7,7,15,27,0.904176
2,456,7,7,15,27,0.997654
3,495,7,7,15,27,1.028556
4,944,7,7,15,27,0.976843


## Filter by Minimum Nodal Member Angle

`pre_filter` receives each `Data` object before it is stored. Returning `False` drops that graph from the processed dataset. Use a different `cache_name` when changing the filter.

In [8]:
def minimum_member_angle_degrees(data):
    coordinates = data.x[:, 2:4]
    vectors_by_node = [[] for _ in range(data.num_nodes)]

    for start, end in data.member_index.t().tolist():
        vector = coordinates[end] - coordinates[start]
        vectors_by_node[start].append(vector)
        vectors_by_node[end].append(-vector)

    smallest = 180.0
    for vectors in vectors_by_node:
        for i, first in enumerate(vectors[:-1]):
            for second in vectors[i + 1:]:
                cosine = torch.dot(first, second) / (first.norm() * second.norm()).clamp_min(1e-12)
                angle = math.degrees(math.acos(float(cosine.clamp(-1.0, 1.0))))
                smallest = min(smallest, angle)
    return smallest

def keep_min_angle(data, degrees=10.0):
    return minimum_member_angle_degrees(data) >= degrees

In [9]:
angle_filtered = TrussBaseDataset(
    root=DATA_ROOT,
    typologies=TYPOLOGIES,
    panels=PANELS,
    seeds=SEEDS,
    max_designs_per_seed=MAX_DESIGNS_PER_SEED,
    plane=PLANE,
    cache_name="tutorial_base_min_angle_10",
    pre_filter=keep_min_angle,
    force_reload=False,
)

print(f"Before filter: {len(cached_dataset):,}")
print(f"After filter:  {len(angle_filtered):,}")

Before filter: 25
After filter:  12


## Scale to a Material and Load Case

The dataset stores coefficients produced under the nominal setup. For a linear elastic truss, a simple scaling is applied toall structural analysis results in a truss by a uniform factor:

- Applied load (without reactions)       → F ← scale · F
- Axial forces  → N ← scale · N
- Displacements → u ← scale · u

`pre_transform` applies this once and stores the scaled result in the processed cache.

In [10]:
def scale_to_material(data, *, scale):
    out = data.clone()

    out.x[:, 4:6] *= scale
    out.x[:, 6:8] *= scale
    out.member_attr[:, 1] *= scale
    out.edge_attr[:, 1] *= scale
    return out

In [11]:
SCALE = 300

scaled_pre = TrussBaseDataset(
    root=DATA_ROOT,
    typologies=TYPOLOGIES,
    panels=PANELS,
    seeds=SEEDS,
    max_designs_per_seed=MAX_DESIGNS_PER_SEED,
    plane=PLANE,
    cache_name="tutorial_base_scaled_steel",
    pre_transform=lambda data: scale_to_material(
        data,
        scale=SCALE,
    ),
    force_reload=False,
)

raw = cached_dataset[0]
scaled = scaled_pre[0]
member_id = int(raw.panels) + 1
node_id = int(raw.panels) + 1
print("Raw member [length, axial]:", raw.member_attr[member_id].tolist())
print("Scaled member:", scaled.member_attr[member_id].tolist())
print("Raw node [force_x, force_z, ux, uz]:", raw.x[node_id, 4:8].tolist())
print("Scaled node:", scaled.x[node_id, 4:8].tolist())

Raw member [length, axial]: [4.140582084655762, -2.7634823322296143]
Scaled member: [82.8116455078125, -165808.9375]
Raw node [force_x, force_z, ux, uz]: [0.0, -0.16710714995861053, 10.00201416015625, -48.38887405395508]
Scaled node: [0.0, -10026.4287109375, 0.012630798853933811, -0.06110670790076256]


## Runtime `transform` Instead of `pre_transform`

`transform` applies every time an item is accessed. The cached processed dataset stays raw, so this is better when you want to test several materials from the same imported cache.

In [12]:
scaled_at_runtime = TrussBaseDataset(
    root=DATA_ROOT,
    typologies=TYPOLOGIES,
    panels=PANELS,
    seeds=SEEDS,
    max_designs_per_seed=MAX_DESIGNS_PER_SEED,
    plane=PLANE,
    cache_name=CACHE_NAME,
    transform=lambda data: scale_to_material(
        data,
        scale=SCALE,
    ),
    force_reload=False,
)

print("Runtime scaled member:", scaled_at_runtime[0].member_attr[member_id].tolist())
print("Raw cache still unchanged:", cached_dataset[0].member_attr[member_id].tolist())

Runtime scaled member: [4.140582084655762, -829.044677734375]
Raw cache still unchanged: [4.140582084655762, -2.7634823322296143]
